# 汽车排序问题

**类别:** 调度

来源: [https://www.hexaly.com/templates/car-sequencing](https://www.hexaly.com/templates/car-sequencing)


## 问题描述

**汽车排序问题** 涉及安排一组汽车的生产顺序。这些汽车并非完全相同,在基本车型的基础上有不同的配置选项可供选择。装配线上有不同的工作站来安装各种选项(空调、天窗等)。这些工作站有最大产能限制,它们最多能处理装配线上一定比例的车辆。因此,必须将汽车排成一个序列,以使每个工作站的产能都不会被超过。例如,如果某个工作站最多能处理装配线上三分之二的车辆,那么在序列中任意连续 3 辆车的窗口内,至多只能有 2 辆车需要该选项。

	

### 学到的要点

- 使用 [列表决策变量](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html) 来表示汽车的序列
- 区分 [结构性约束与第一优先级目标](https://www.hexaly.com/docs/last/modelingprinciples/modelingprinciples.html#distinguish-constraints-from-first-priority-objectives)
- 使用 [非线性算子](https://www.hexaly.com/docs/last/mathematicaloperators/operatorsreference.html#table-of-available-operators-and-functions) 来计算违规次数


## 数据

我们提供的汽车排序问题实例来自 [CSPLib](http://www.csplib.org/Problems/prob001/)。数据文件的格式如下:

- 第 1 行: 车辆数量、选项数量、类别数量
- 第 2 行: 对于每个选项,一个块中包含该选项的最大车辆数
- 第 3 行: 对于每个选项,该最大车辆数所对应的块大小
- 然后,对于每个类别:

- 类别的索引
- 该类别中的车辆数量
- 对于每个选项,该类别是否需要它(1 或 0)。


## 模型

汽车排序问题的 Hexaly 模型使用一个 [列表决策变量](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html) 来表示汽车的序列。列表的第 i 个元素对应于第 i 辆要生产的汽车的索引。使用 [**partition**](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html#n-ary-operators) 算子,可以确保待生产的每辆汽车都出现在装配线上。

根据这个序列,我们可以对每个选项和装配线上的每个位置,计算出从该位置开始的窗口中具有该选项的车辆数量。然后我们可以推导出每个选项和每个窗口的违规数量。

尽管这个问题是一个纯可行性问题,我们仍然选择添加一个目标,即使所有选项和所有窗口的产能违规次数之和最小化。事实上,无产能违规更像是“业务”约束,而不是结构性约束。如果出现少量违规,装配线会减速但仍可继续运行。相反,在装配线上的某个位置同时放置两辆车在物理上是不可能的:这是一个结构性约束。有关高优先级目标和硬约束之间区别的更多信息,请参阅文档中的 [此章节](https://www.hexaly.com/docs/last/modelingprinciples/modelingprinciples.html#distinguish-constraints-from-first-priority-objectives)。


## Python 实现


In [ ]:
# Copyright (c) Hexaly. Permission is hereby granted to use, copy,
# and modify this code for applications developed with Hexaly.
import hexaly.optimizer
import sys

#
# Read instance data
#

def read_integers(filename):
    with open(filename) as f:
        return [int(elem) for elem in f.read().split()]

def read_instance(instance_file):
    file_it = iter(read_integers(instance_file))
    nb_positions = next(file_it)
    nb_options = next(file_it)
    nb_classes = next(file_it)
    max_cars_per_window = [next(file_it) for i in range(nb_options)]
    window_size = [next(file_it) for i in range(nb_options)]
    nb_cars = []
    options = []
    initial_sequence = []

    for c in range(nb_classes):
        next(file_it)  # Note: index of class is read but not used
        nb_cars.append(next(file_it))
        options.append([next(file_it) == 1 for i in range(nb_options)])
        [initial_sequence.append(c) for p in range(nb_cars[c])]

    return nb_positions, nb_options, max_cars_per_window, window_size, options, \
        initial_sequence


def main(instance_file, output_file, time_limit):
    nb_positions, nb_options, max_cars_per_window, window_size, options, \
        initial_sequence = read_instance(instance_file)

    with hexaly.optimizer.HexalyOptimizer() as optimizer:
        #
        # Declare the optimization model
        #
        model = optimizer.model

        # sequence[i] = j if class initially planned on position j is produced on position i
        sequence = model.list(nb_positions)

        # sequence is a permutation of the initial production plan, all indexes must
        # appear exactly once
        model.constraint(model.partition(sequence))

        # Create Hexaly arrays to be able to access them with "at" operators
        initial_array = model.array(initial_sequence)
        option_array = model.array(options)

        # Number of cars with option o in each window
        nb_cars_windows = [None] * nb_options
        for o in range(nb_options):
            nb_cars_windows[o] = [None] * nb_positions
            for j in range(nb_positions - window_size[o] + 1):
                nb_cars_windows[o][j] = model.sum()
                for k in range(window_size[o]):
                    class_at_position = initial_array[sequence[j + k]]
                    nb_cars_windows[o][j].add_operand(option_array[class_at_position][o])

        # Number of violations of option o capacity in each window
        nb_violations_windows = [None] * nb_options
        for o in range(nb_options):
            nb_violations_windows[o] = [None] * nb_positions
            for p in range(nb_positions - window_size[o] + 1):
                nb_violations_windows[o][p] = model.max(
                    nb_cars_windows[o][p] - max_cars_per_window[o], 0)

        # Minimize the sum of violations for all options and all windows
        total_violations = model.sum(
            nb_violations_windows[o][p]
            for o in range(nb_options) for p in range(nb_positions - window_size[o] + 1))
        model.minimize(total_violations)

        model.close()

        # Set the initial solution
        sequence.get_value().clear()
        for p in range(nb_positions):
            sequence.get_value().add(p)

        # Parameterize the optimizer
        optimizer.param.time_limit = time_limit

        optimizer.solve()

        #
        # Write the solution in a file with the following format:
        # - 1st line: value of the objective;
        # - 2nd line: for each position p, index of class at positions p.
        #
        if output_file is not None:
            with open(output_file, 'w') as f:
                f.write("%d\n" % total_violations.value)
                for p in range(nb_positions):
                    f.write("%d " % initial_sequence[sequence.value[p]])

                f.write("\n")


if __name__ == '__main__':
    if len(sys.argv) < 2:
        print("Usage: python car_sequencing.py instance_file [output_file] [time_limit]")
        sys.exit(1)

    instance_file = sys.argv[1]
    output_file = sys.argv[2] if len(sys.argv) >= 3 else None
    time_limit = int(sys.argv[3]) if len(sys.argv) >= 4 else 60
    main(instance_file, output_file, time_limit)
